# Lab 20: How to evaluate and trace an agent

**Notebook 1 of 2: test, trace and judge a library helper**

## Goal

Test a small agent the way a team tests one before it goes to work: with questions whose right answers we write down
first, a trace of every step the agent took, and a second model that marks answers our code cannot check. The agent
is a library's opening-hours helper. Three of its answers pass. The fourth question is about a public holiday that the
opening hours never mention, and the helper answers it with a confident "Yes". The trace on the MLflow page shows that
the helper read the hours and answered beyond them. Last, a judge marks a wrong quiz answer as right, because the
evidence we gave it came from a prank website. This notebook uses everyday examples; notebook 2 does the same work at
the Chinook support desk.

## What you will learn

- How to write test cases with an answer key the model never sees, and check each reply in code.
- How to start the MLflow page on your own machine and record every agent run as a trace with one line.
- How to read a trace as a tree of spans: the run, the agent, each model call and each tool call, with their inputs,
  outputs and time.
- How to use a failing test and its trace together: the test says the answer is wrong, and the trace shows what the
  agent read before it answered.
- How a judge (a second model call with a rubric) marks an answer, and why its verdict follows the evidence we give
  it.

## Setup

The first cell is the same in every lab: it finds the repository folder, reads the model name from `models.env`, and
takes the key from `.env` without printing it (lab 00 explains it line by line).

In [1]:
from pathlib import Path
import os
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


One more line of setup: a 60-second limit on every model request the Agents SDK makes, so a request that gets stuck
fails quickly instead of freezing a cell. (The default is ten minutes.)

In [2]:
from openai import AsyncOpenAI
from agents import set_default_openai_client

set_default_openai_client(AsyncOpenAI(timeout=60, max_retries=2))

---

## Part A: Test the helper with questions whose answers we wrote down first

The Riverside Library wants a helper on its website that answers questions about opening times. Before anyone relies
on it, we want evidence that it answers correctly, and a way to collect that evidence again after every change. This
part collects it by hand: a few questions, the answer each one should get, and a check in code.

### The helper

The library's opening hours are one short note:

In [3]:
HOURS = (
    "Riverside Library opening hours: Monday to Friday 9:00 to 20:00. "
    "Saturday 10:00 to 16:00. Sunday closed."
)

The helper is an agent with one tool, `get_opening_hours`, which returns that note. We build it with the OpenAI
Agents SDK:

- `@function_tool` turns the Python function into a tool the model can ask for. The tool's name is the function's
  name, and its description is the docstring.
- `Agent` holds the instructions, the tools and the model.
- `ModelSettings(reasoning=Reasoning(effort=EFFORT))` sets the reasoning effort from `models.env`.

The library asked for short, direct answers, so the last sentence of the instructions asks for one sentence, with Yes
or No first when the question asks one.

In [4]:
from agents import Agent, Runner, ModelSettings, function_tool
from openai.types.shared import Reasoning


@function_tool
def get_opening_hours() -> str:
    """Read the library's opening hours."""
    return HOURS


helper = Agent(
    name="Library helper",
    instructions=(
        "You answer questions about the Riverside Library. Use get_opening_hours for opening times. "
        "Answer in one short sentence, and start with Yes or No when the question asks one."
    ),
    tools=[get_opening_hours],
    model=MODEL,
    model_settings=ModelSettings(reasoning=Reasoning(effort=EFFORT)),
)

We ask one question and read the reply. `Runner.run` runs the agent's loop: the model asks for the tool, the SDK runs it and sends the result back, and the model writes the answer. `result.final_output` is that answer. `Runner.run` is asynchronous, so we `await` it: `await` waits until the run has finished, and Jupyter allows it directly in a cell.

In [5]:
result = await Runner.run(helper, "What time do you close on Saturday?")
print(result.final_output)

Riverside Library closes at 4:00 p.m. on Saturday.


The helper says the library closes at 4:00 p.m. on Saturday. The note says Saturday 10:00 to 16:00, so the answer is right, written on the 12-hour clock.

### Turn the right answer into a check

We can see that this reply is right. To check it again tomorrow, or after a change to the instructions, we write the
right answer down as a **test case**: the question, and a pattern that every right reply contains. The pattern is the
**answer key**. It stays in our code. Only the question is sent to the helper.

The pattern is a regular expression, read by Python's `re` module:

- `16:00|4(:00)? ?p\.?m` means "16:00, or 4 pm written in any of the usual ways". `|` separates the alternatives,
  `(:00)?` and ` ?` make a part optional, and `\.?` allows a full stop after the p.
- `re.search(pattern, text, re.IGNORECASE)` looks for the pattern anywhere in the text, ignoring capitals, and
  returns a match or `None`. `bool(...)` turns that into `True` or `False`.

In [6]:
import re

case = {"question": "What time do you close on Saturday?", "expect": r"16:00|4(:00)? ?p\.?m"}

print(bool(re.search(case["expect"], result.final_output, re.IGNORECASE)))

True


`True`: the reply passes. A pass means that the pattern appears in the reply, and nothing more. A reply that said "we
do not close at 4 pm" would pass too, so a test set is read, not only counted.

### Run a small test set

A **test set** holds several cases, each chosen for a reason. Ours has three: the Saturday question, a question whose
right answer is "no" (Sunday), and another day of the week. Every answer key is written before anything runs.

In [7]:
CASES = [
    {"question": "What time do you close on Saturday?", "expect": r"16:00|4(:00)? ?p\.?m"},
    {"question": "Are you open on Sunday?", "expect": r"\bno\b|closed"},
    {"question": "When do you open on Wednesday?", "expect": r"\b9(:00)?\b"},
]

`\b` marks the edge of a word, so `\bno\b` matches the word "no" but not the "no" inside "not", and `\b9(:00)?\b`
matches 9 or 9:00 but not 19:00.

The loop sends each question to the helper and checks the reply against that case's own answer key. It prints ✅ or ❌,
then the question and the reply. We put it in a function, `run_cases`, because we will run the set again. A function that uses `await` inside it is written `async def`, and it is called with `await` too.

In [8]:
async def run_cases(agent, cases):
    for case in cases:
        result = await Runner.run(agent, case["question"])
        passed = bool(re.search(case["expect"], result.final_output, re.IGNORECASE))
        print("✅" if passed else "❌", case["question"], "->", result.final_output)


await run_cases(helper, CASES)

✅ What time do you close on Saturday? -> Riverside Library closes at 4:00 p.m. on Saturday.


✅ Are you open on Sunday? -> No, Riverside Library is closed on Sundays.


✅ When do you open on Wednesday? -> Riverside Library opens at 9:00 on Wednesday.


Three ticks. Each reply matches its answer key: 4:00 p.m. on Saturday, closed on Sunday, 9:00 on Wednesday.

### A question the opening hours do not answer

The note gives the hours for each day of the week and says nothing about public holidays. Next Monday is a public
holiday. A right answer says that the hours do not cover it, so the answer key looks for phrases such as "don't
specify", "do not cover" or "can't confirm".

**Before you run the next cell:** what should the helper answer, and what do you expect it to answer?

In [9]:
holiday = {
    "question": "Are you open on the public holiday next Monday?",
    "expect": r"(don.t|doesn.t|do not|does not|can.t|cannot|not) (say|specify|cover|list|mention|confirm|know)",
}
CASES.append(holiday)

await run_cases(helper, CASES)

✅ What time do you close on Saturday? -> Riverside Library closes at 4:00 p.m. on Saturday.


✅ Are you open on Sunday? -> No, Riverside Library is closed on Sunday.


✅ When do you open on Wednesday? -> Riverside Library opens at 9:00 on Wednesday.


❌ Are you open on the public holiday next Monday? -> Yes, Riverside Library is open next Monday from 9:00 to 20:00.


Three passes again, and one failure. To the holiday question the helper answered *Yes, Riverside Library is open next Monday from 9:00 to 20:00.* The note never mentions public holidays, so the helper stated something the note does not support, and nothing in the reply says it is a guess.

Notice that the three earlier cases ran again. A test set is run whole every time, so that a change which fixes one
case cannot quietly break another.

The reply leaves a question open: did the helper read the hours and still say yes, or did it never call the tool? The reply alone cannot tell us. A trace records every tool call the helper made, so it can.

**Checkpoint A.** Three ✅ lines, then the same three again and one ❌: the helper answers the public holiday question
with Yes and Monday's usual hours. Your replies will be worded differently; check the ticks and the holiday answer.

---

## Part B: Trace a run and read its spans on the MLflow page

A **trace** is the record of one run of the agent. It is made of **spans**: one span for each step (the run, the
agent, each model call, each tool call), with its inputs, its outputs and how long it took. A span can sit inside
another span, so a trace reads as a tree.

We record traces with [MLflow](https://mlflow.org/docs/latest/genai/tracing/), an open-source platform that keeps
them in a database on our own machine and shows them on a web page. No account is needed. There are two steps: start
the page, then switch recording on.

### Start the MLflow page

The next cell starts the MLflow server as a separate program, the same way you would start it from a terminal:

- `sys.executable -m mlflow server` runs MLflow from the same Python environment as this notebook.
- `--host 127.0.0.1 --port 5020` serves the page on this machine only, at http://127.0.0.1:5020.
- `--backend-store-uri sqlite:///outputs/mlflow.db` keeps every trace in one SQLite file in this lab's `outputs/`
  folder, which git ignores.
- `page_is_up()` asks the server's `/health` address whether it answers. The cell waits until it does; the first start
  takes about half a minute while MLflow sets up its database. If the server stops instead (for example because
  another program holds port 5020), the `assert` says so.

If the page is already running, from an earlier run of this notebook, the cell uses it.

In [10]:
import subprocess
import sys
import time
import requests

PORT = 5020
URL = f"http://127.0.0.1:{PORT}"
LAB = ROOT / "20_Evals_And_Tracing"
(LAB / "outputs").mkdir(exist_ok=True)


def page_is_up():
    try:
        return requests.get(URL + "/health", timeout=1).ok
    except requests.ConnectionError:
        return False


server = None
if not page_is_up():
    server = subprocess.Popen(
        [sys.executable, "-m", "mlflow", "server", "--host", "127.0.0.1", "--port", str(PORT),
         "--backend-store-uri", "sqlite:///outputs/mlflow.db"],
        cwd=LAB, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    )
    while not page_is_up():
        assert server.poll() is None, f"The MLflow server stopped. Is port {PORT} used by another program?"
        time.sleep(1)
print("The MLflow page is running at", URL)

The MLflow page is running at http://127.0.0.1:5020


From a terminal, the same page starts with `uv run mlflow server --host 127.0.0.1 --port 5020 --backend-store-uri
sqlite:///outputs/mlflow.db`, run in the `20_Evals_And_Tracing` folder (Ctrl+C stops it). On a remote machine or in
Codespaces, open the address VS Code forwards for port 5020.

### Switch recording on

Three lines connect this notebook to the page:

- `set_tracking_uri` points MLflow at the server.
- `set_experiment` names the group our traces go in, "Library helper". MLflow creates it the first time.
- `mlflow.openai.autolog()` records every call of the OpenAI client and every Agents SDK run in this kernel from now
  on. The helper's code does not change.

Two more lines keep the output plain. The first hides a notice about progress bars that `import mlflow` prints, so it
comes before the import. The last, `disable_notebook_display()`, stops MLflow drawing its own trace viewer under each
cell: we read the traces on the page instead, and as printed text here.

In [11]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # a progress-bar notice we do not need
import mlflow

mlflow.set_tracking_uri(URL)
mlflow.set_experiment("Library helper")
mlflow.openai.autolog()
mlflow.tracing.disable_notebook_display()

2026/11/12 16:33:54 INFO mlflow.tracking.fluent: Experiment with name 'Library helper' does not exist. Creating a new experiment.


**Important:** with `autolog` on, the Agents SDK no longer sends its traces to the Traces page of your OpenAI
account. MLflow takes over that job, so the traces stay on this machine.

### Ask the holiday question again

The same agent and the same question. After the run, `mlflow.get_last_active_trace_id()` gives the id of the trace
this run made, and `mlflow.get_trace` reads the whole trace back from the server. MLflow sends traces to the server
in the background; `flush=True` waits until this one has arrived.

In [12]:
result = await Runner.run(helper, holiday["question"])
print(result.final_output)

trace = mlflow.get_trace(mlflow.get_last_active_trace_id(), flush=True)
print("trace id:", trace.info.trace_id)

Yes, Riverside Library is open next Monday from 9:00 to 20:00.


trace id: tr-002a8c538672c25c9954e3ddb8f3192d


The same answer as in Part A: *Yes, Riverside Library is open next Monday from 9:00 to 20:00.* The trace id, `tr-` and 32 characters, names this run's trace on the server; yours will be different.

### Read the trace as a tree

`trace.data.spans` lists every span of the trace in the order they started. Each span carries a `span_id` and a
`parent_id`, the id of the span it sits inside (the outermost span has `None`). To print the tree, we indent each span
by its depth:

1. `by_id` maps each span's id to the span, so we can find a parent from its id.
2. `depth(span)` is 0 for the outermost span; otherwise it is one more than its parent's depth.
3. For each span we print its name, its type (`span_type`) and its duration. Start and end times are in nanoseconds,
   so dividing by 1,000,000 gives milliseconds.

In [13]:
by_id = {span.span_id: span for span in trace.data.spans}


def depth(span):
    return 0 if span.parent_id is None else 1 + depth(by_id[span.parent_id])


for span in trace.data.spans:
    ms = (span.end_time_ns - span.start_time_ns) / 1_000_000
    print("    " * depth(span) + f"{span.name}  ({span.span_type}, {ms:.0f} ms)")

AgentRunner.run  (AGENT, 4549 ms)
    Agent workflow  (CHAIN, 4388 ms)
        Library helper  (AGENT, 4387 ms)
            Unknown  (CHAIN, 3315 ms)
                Response  (CHAT_MODEL, 3302 ms)
                    AsyncResponses  (CHAT_MODEL, 3298 ms)
                get_opening_hours  (TOOL, 4 ms)
            Unknown  (CHAIN, 1071 ms)
                Response  (CHAT_MODEL, 1057 ms)
                    AsyncResponses  (CHAT_MODEL, 1055 ms)


Read the tree from the top:

| Span | What it records |
|---|---|
| `AgentRunner.run` | the whole run: our call to `Runner.run` |
| `Agent workflow` | the Agents SDK's own record of the run |
| `Library helper` | our agent |
| `Unknown` | one turn of the agent's loop. The SDK records each turn, and MLflow 3.18 shows that kind of span without a name |
| `Response`, and `AsyncResponses` inside it | one model call, recorded twice: by the SDK, and by the OpenAI client underneath it, whose span carries the token counts |
| `get_opening_hours` | the tool call |

The run took two turns. In the first, the model asked for `get_opening_hours` and the SDK ran it (4 ms); in the second, the model wrote the answer. Almost all of the 4.5 seconds went to the two model calls (3.3 and 1.1 seconds).

### What the helper read

`trace.search_spans(span_type="TOOL")` returns the tool spans of the trace. A span's `inputs` and `outputs` are what
the step received and what it returned.

In [14]:
tool_span = trace.search_spans(span_type="TOOL")[0]
print("tool:   ", tool_span.name)
print("inputs: ", tool_span.inputs)
print("outputs:", tool_span.outputs)

tool:    get_opening_hours
inputs:  {}
outputs: Riverside Library opening hours: Monday to Friday 9:00 to 20:00. Saturday 10:00 to 16:00. Sunday closed.


The tool took no arguments (`{}`) and returned the opening hours, word for word. Nothing in them mentions public holidays. So the helper did read the hours, and the "Yes" came from the model, not from the note. The instruction to start with Yes or No asked for a direct answer, and the model gave one.

### The same trace on the MLflow page

Open http://127.0.0.1:5020 in your browser.

1. Click **Experiments**, then **Library helper**.
2. Open **Traces**. Each row is one run, newest first; the top row is the holiday question.
3. Click the row. The spans are on the left, as the same tree; click `get_opening_hours` to see its inputs and outputs,
   or an `AsyncResponses` span to see the messages sent to the model and the tokens it used.

![The MLflow page with the holiday trace open: the span tree on the left (AgentRunner.run, Agent workflow, Library helper, two Unknown turns with their Response and AsyncResponses spans, and get_opening_hours), and on the right the output of get_opening_hours, the opening hours note](resources/lab20_mlflow_trace.png)

**Key lesson:** a trace shows what the agent did, not whether its answer is right. The test case found the wrong
answer, and the trace showed what happened before it: the helper read the opening hours and answered beyond them.

**Checkpoint B.** The page answers at http://127.0.0.1:5020. The tree has `AgentRunner.run` at the top, the agent
inside it, two turns, two model calls and one `get_opening_hours` call. The tool span's output is the opening hours
note, with no word about holidays.

---

## Part C: Mark an answer with a second model, and see that the verdict follows the evidence

Our answer keys work because each right answer fits a pattern. Many answers do not: is an explanation correct, does a
reply stick to the facts it was given? For those, teams use a **judge**: a second model call that reads a rubric (the
marking rules), the evidence and the answer, and returns a verdict.

A quiz app is the smallest example. A player answered *The Danube is the longest river in the world.* The app finds
one line of evidence on the web and asks the judge to mark the answer.

`judge(evidence)` makes one call on the Responses API with the plain OpenAI client:

- `instructions` is the rubric: decide from the evidence, reply CORRECT or WRONG and one sentence that says why.
- `input` puts the question, the evidence and the answer in one message.
- `reasoning` passes the reasoning effort from `models.env`; `omit` leaves the field out when `models.env` sets no
  effort.
- `output_text` is the judge's reply.

Autolog records the judge's calls as well, so they appear on the MLflow page beside the helper's runs.

In [15]:
from openai import OpenAI, omit

client = OpenAI(timeout=60)

RUBRIC = (
    "You mark answers to a quiz. Decide whether the answer is correct according to the evidence. "
    "Reply CORRECT or WRONG, then one short sentence that says why."
)
QUESTION = "Which river is the longest in the world?"
ANSWER = "The Danube is the longest river in the world."


def judge(evidence):
    response = client.responses.create(
        model=MODEL,
        instructions=RUBRIC,
        input=f"Question: {QUESTION}\nEvidence: {evidence}\nAnswer: {ANSWER}",
        reasoning={"effort": EFFORT or omit},
    )
    return response.output_text

### Evidence from a prank website

The app's web search found a "fact of the day" page that publishes jokes:

In [16]:
PRANK_SITE = "Fact of the day: the Danube is the longest river in the world, at 9,000 km."

print(judge(PRANK_SITE))

CORRECT — The evidence says the Danube is the longest river in the world.


CORRECT. The judge marked the wrong answer right, and its reason points to the evidence: *the Danube is the longest river in the world*. The rubric told it to decide according to the evidence, and it did.

### Evidence from an atlas

Now the same rubric and the same answer, with a line from an atlas as the evidence:

In [17]:
ATLAS = "The Nile is the longest river in the world, at about 6,650 km."

print(judge(ATLAS))

WRONG — The evidence identifies the Nile, not the Danube, as the longest river in the world.


WRONG, and the reason names the Nile.

**Lesson to Remember**

| Evidence | Verdict on *The Danube is the longest river in the world.* |
|---|---|
| a prank website: the Danube, at 9,000 km | CORRECT |
| an atlas: the Nile, at about 6,650 km | WRONG |

Notice that the rubric and the answer did not change. Only the evidence changed, and the verdict followed it. **A
judge checks an answer against the evidence we give it. It cannot tell whether that evidence is true.** So choosing
the evidence is part of writing a judge. Notebook 2 makes that choice at the support desk, where a customer's ticket
and the billing record disagree about a refund.

**Checkpoint C.** With the prank line the judge replies CORRECT, and with the atlas line it replies WRONG, both with a
reason that points to the evidence.

---

## What you can do now

- Write test cases with an answer key the model never sees, and check every reply in code (Part A).
- Start the MLflow page on your machine and record agent runs as traces with `mlflow.openai.autolog()` (Part B).
- Read a trace as a tree of spans, and find what a tool returned and what a model call used (Part B).
- Use a failing test and its trace together to say what went wrong and where (Parts A and B).
- Write a small judge, and explain why its verdict depends on the evidence it reads (Part C).

## Troubleshooting

| What you see | Why | Fix |
|---|---|---|
| `AssertionError: The MLflow server stopped. Is port 5020 used by another program?` | another program holds port 5020, so the MLflow server could not start | stop that program, or set `PORT` to a free port (5021) in the cell and open that address instead |
| `AttributeError: 'NoneType' object has no attribute 'info'` after `get_trace` | the trace had not reached the server yet when we asked for it | keep `flush=True` in `mlflow.get_trace(..., flush=True)` |
| `MlflowException: API request to http://127.0.0.1:5020/... failed with exception ... Max retries exceeded`, after a wait of several minutes | the MLflow page is not running: it was stopped, or the kernel restarted after the cell that started it | run the cell that starts the page, then run your cell again |
| The page opens but shows no traces | the page lists the experiments; ours is "Library helper" | click **Experiments**, then **Library helper**, then **Traces** |
| On a remote machine the address does not open | the page listens on that machine only (`127.0.0.1`) | open the address VS Code forwards for port 5020 (the **Ports** tab) |

## Challenge

Fix the holiday answer without breaking the other three cases.

1. Make a new version of the helper with one more sentence in its instructions: what to do when the opening hours do
   not cover a day or date. `helper.clone(instructions=...)` copies the agent with new instructions.
2. Run all four cases on the new version with `run_cases`.
3. On the MLflow page, open the new holiday trace and compare its answer with the old one.

Expected: four ✅ lines. The holiday reply says that the opening hours do not cover public holidays, and the Saturday,
Sunday and Wednesday answers still pass.

The worked solution, run with its outputs, is in `solved/20a_meet_evals_and_tracing_solution.ipynb`.

In [19]:
# Your new version of the helper, and the four cases run on it

---

## Stop the MLflow page

When you have finished with the page, run the next cell. The traces stay in `outputs/mlflow.db`: run the cell that
starts the page again, and they are all still there.

In [18]:
if server is not None:
    server.terminate()
    server.wait()
    print("The MLflow page has stopped.")

The MLflow page has stopped.


## Resources

- [MLflow tracing](https://mlflow.org/docs/latest/genai/tracing/): what a trace and a span record
- [MLflow: tracing the OpenAI Agents SDK](https://mlflow.org/docs/latest/genai/tracing/integrations/listing/openai-agent/):
  `mlflow.openai.autolog()` for agents, tools and handoffs
- [MLflow: set up the server](https://mlflow.org/docs/latest/genai/getting-started/connect-environment/): starting the
  page on your own machine
- [MLflow: custom judges with `make_judge`](https://mlflow.org/docs/latest/genai/eval-monitor/scorers/llm-judge/custom-judges/):
  a judge's instructions, its placeholders and its verdict type
- [OpenAI: evaluation best practices](https://developers.openai.com/api/docs/guides/evaluation-best-practices): test
  sets, answer keys and judges
- [OpenAI Agents SDK: tracing](https://openai.github.io/openai-agents-python/tracing/): the spans the SDK records
- [Python `re`](https://docs.python.org/3/library/re.html): regular expressions

Next: [notebook 2, evaluation and tracing at the Chinook desk](20b_evals_and_tracing_at_the_desk.ipynb).